# O*NET Data Exploration and Cleaning

This notebook loads the available O*NET files from `data/raw/`, profiles each workbook, standardises SOC codes, and builds two cleaned outputs in `data/processed/`: `master_occupations.csv` and `master_skills.csv`.

## Section 1 — Imports and setup

Import libraries, configure display settings, and define filesystem paths using `pathlib` so the notebook works on any machine.

In [23]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings for pandas dataframes
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}' if pd.notnull(x) else '')

# Define paths relative to repository root. Adjust this block if the notebook is moved.
ROOT = Path('..').resolve() if Path('..').exists() else Path('.').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

ROOT: ..
RAW_DIR: ../data/raw
PROCESSED_DIR: ../data/processed


## Section 2 — Load and profile each file

Load each workbook into a named dataframe and use a reusable profiling function to inspect shape, columns, missing values, and sample rows.

In [22]:
def profile_df(df: pd.DataFrame, name: str) -> None:
    print(f'--- Profile: {name} ---')
    print('Shape:', df.shape)
    print('Columns:', df.columns.tolist())
    print('Null counts:')
    print(df.isnull().sum().to_string())
    print('Head:')
    display(df.head(3))
    print()


def load_excel(filename: str, sheet_name=0):
    path = RAW_DIR / filename
    # Try direct path first
    if path.exists():
        return pd.read_excel(path, sheet_name=sheet_name)

    # Recursive search under RAW_DIR for exact filename
    matches = list(RAW_DIR.rglob(filename))

    # Try URL-decoded variant (handles %20 spaces)
    if not matches:
        try:
            from urllib.parse import unquote
            alt = unquote(filename)
            if alt != filename:
                matches = list(RAW_DIR.rglob(alt))
        except Exception:
            pass

    # Case-insensitive filename match
    if not matches:
        name_lower = filename.lower()
        matches = [p for p in RAW_DIR.rglob('*') if p.is_file() and p.name.lower() == name_lower]

    # Match by basename without extension
    if not matches:
        base_no_ext = Path(filename).stem.lower()
        matches = [p for p in RAW_DIR.rglob('*') if p.is_file() and Path(p).stem.lower() == base_no_ext]

    if matches:
        chosen = matches[0]
        print(f"Loading {filename} from {chosen}")
        return pd.read_excel(chosen, sheet_name=sheet_name)

    # Provide helpful error listing available files
    available = [str(p.relative_to(RAW_DIR)) for p in RAW_DIR.rglob('*') if p.is_file()]
    sample_list = '\n'.join(available[:200]) if available else '(no files found under data/raw/)'
    raise FileNotFoundError(f'Expected file not found: {path}\nSearched recursively under {RAW_DIR}.\nAvailable files (sample):\n{sample_list}')


# Helper: try several candidate filenames for a logical dataset
def try_load_variants(candidates, logical_name):
    last_err = None
    for cand in candidates:
        try:
            df = load_excel(cand)
            print(f"Loaded {logical_name} from candidate: {cand}")
            return df
        except FileNotFoundError as e:
            last_err = e
            continue
    # If we reach here, re-raise the last error with added context
    raise FileNotFoundError(f"Could not find any of the expected files for {logical_name}. Tried: {candidates}\n{last_err}")

# Candidate lists (ordered by likely match)
occ_candidates = ['Occupation Data.xlsx', 'Occupation%20Data.xlsx', 'Occupation Data.xls', 'Occupation%20Data.xls']
skills_candidates = ['Skills.xlsx', 'Essential Skills.xlsx', 'Essential%20Skills.xlsx', 'Software Skills.xlsx', 'Software%20Skills.xlsx', 'Transferable Skills.xlsx', 'Transferable%20Skills.xlsx']
knowledge_candidates = ['Knowledge.xlsx', 'Knowledge%20.xlsx']
education_candidates = ['Education Training and Experience.xlsx', 'Education%20Training%20and%20Experience.xlsx', 'Training and Experience.xlsx', 'Training%20and%20Experience.xlsx']
activities_candidates = ['Work Activities.xlsx', 'Work%20Activities.xlsx']
styles_candidates = ['Work Styles.xlsx', 'Work%20Styles.xlsx']
abilities_candidates = ['Abilities.xlsx', 'Abilities%20to%20Work%20Activities.xlsx', 'Abilities%20to%20Work%20Context.xlsx']

# Load each logical dataset using candidates
df_occupations = try_load_variants(occ_candidates, 'occupations')
df_skills = try_load_variants(skills_candidates, 'skills')
df_knowledge = try_load_variants(knowledge_candidates, 'knowledge')
df_education = try_load_variants(education_candidates, 'education/training')
df_activities = try_load_variants(activities_candidates, 'work activities')
df_styles = try_load_variants(styles_candidates, 'work styles')
df_abilities = try_load_variants(abilities_candidates, 'abilities')

# Profile each dataframe
profile_df(df_occupations, 'df_occupations')
profile_df(df_skills, 'df_skills')
profile_df(df_knowledge, 'df_knowledge')
profile_df(df_education, 'df_education')
profile_df(df_activities, 'df_activities')
profile_df(df_styles, 'df_styles')
profile_df(df_abilities, 'df_abilities')


Loading Occupation Data.xlsx from ../data/raw/onet/Occupation Data.xlsx
Loaded occupations from candidate: Occupation Data.xlsx
Loading Essential Skills.xlsx from ../data/raw/onet/Essential Skills.xlsx
Loaded skills from candidate: Essential Skills.xlsx
Loading Knowledge.xlsx from ../data/raw/onet/Knowledge.xlsx
Loaded knowledge from candidate: Knowledge.xlsx
Loading Training and Experience.xlsx from ../data/raw/onet/Training and Experience.xlsx
Loaded education/training from candidate: Training and Experience.xlsx
Loading Work Activities.xlsx from ../data/raw/onet/Work Activities.xlsx
Loaded work activities from candidate: Work Activities.xlsx
Loading Work Styles.xlsx from ../data/raw/onet/Work Styles.xlsx
Loaded work styles from candidate: Work Styles.xlsx
Loading Abilities.xlsx from ../data/raw/onet/Abilities.xlsx
Loaded abilities from candidate: Abilities.xlsx
--- Profile: df_occupations ---
Shape: (1016, 3)
Columns: ['O*NET-SOC Code', 'Title', 'Description']
Null counts:
O*NET-SOC

,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide overall direction of companies or private and publi...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, shareholders, customers, and employees to address su..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of public or private sector organizations, overseeing..."



--- Profile: df_skills ---
Shape: (17880, 15)
Columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']
Null counts:
O*NET-SOC Code           0
Title                    0
Element ID               0
Element Name             0
Scale ID                 0
Scale Name               0
Data Value               0
N                        0
Standard Error           0
Lower CI Bound           0
Upper CI Bound           0
Recommend Suppress       0
Not Relevant          8940
Date                     0
Domain Source            0
Head:


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.12,8,0.12,3.88,4.37,N,NaN,08/2023,Analyst
1,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,LV,Level,4.62,8,0.18,4.27,4.98,N,N,08/2023,Analyst
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.00,8,0.00,4.00,4.00,N,NaN,08/2023,Analyst



--- Profile: df_knowledge ---
Shape: (59004, 15)
Columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']
Null counts:
O*NET-SOC Code            0
Title                     0
Element ID                0
Element Name              0
Scale ID                  0
Scale Name                0
Data Value                0
N                      1056
Standard Error        14916
Lower CI Bound        15604
Upper CI Bound        15604
Recommend Suppress    13926
Not Relevant          29502
Date                      0
Domain Source             0
Head:


...

,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,4.A.1.a.1,Getting Information,IM,Importance,4.56,29.00,0.16,4.24,4.88,N,NaN,08/2023,Incumbent
1,11-1011.00,Chief Executives,4.A.1.a.1,Getting Information,LV,Level,4.89,30.00,0.17,4.54,5.25,N,N,08/2023,Incumbent
2,11-1011.00,Chief Executives,4.A.1.a.2,"Monitoring Processes, Materials, or Surroundings",IM,Importance,4.25,30.00,0.21,3.81,4.68,N,NaN,08/2023,Incumbent



--- Profile: df_styles ---
Shape: (37422, 9)
Columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'Date', 'Domain Source']
Null counts:
O*NET-SOC Code    0
Title             0
Element ID        0
Element Name      0
Scale ID          0
Scale Name        0
Data Value        0
Date              0
Domain Source     0
Head:


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,Date,Domain Source
0,11-1011.00,Chief Executives,1.D.1.a,Innovation,DR,Distinctiveness Rank,7.00,12/2025,AI/Expert
1,11-1011.00,Chief Executives,1.D.1.a,Innovation,WI,Work Styles Impact,2.30,12/2025,AI/Expert
2,11-1011.00,Chief Executives,1.D.1.b,Achievement Orientation,DR,Distinctiveness Rank,0.00,12/2025,AI/Expert



--- Profile: df_abilities ---
Shape: (92976, 15)
Columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']
Null counts:
O*NET-SOC Code            0
Title                     0
Element ID                0
Element Name              0
Scale ID                  0
Scale Name                0
Data Value                0
N                         0
Standard Error            0
Lower CI Bound            0
Upper CI Bound            0
Recommend Suppress        0
Not Relevant          46488
Date                      0
Domain Source             0
Head:


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,1.A.1.a.1,Oral Comprehension,IM,Importance,4.62,8,0.18,4.27,4.98,N,NaN,08/2023,Analyst
1,11-1011.00,Chief Executives,1.A.1.a.1,Oral Comprehension,LV,Level,4.88,8,0.12,4.63,5.12,N,N,08/2023,Analyst
2,11-1011.00,Chief Executives,1.A.1.a.2,Written Comprehension,IM,Importance,4.25,8,0.16,3.93,4.57,N,NaN,08/2023,Analyst


## Section 3 — Standardise SOC codes

Define a function to clean SOC codes and apply it consistently across all dataframes.

In [24]:

def clean_soc(code):
    # Preserve actual missing values and normalise strings
    if pd.isna(code):
        return np.nan
    code_str = str(code).strip()
    if code_str.lower() in ('nan', 'none', ''):
        return np.nan
    if '.' in code_str:
        code_str = code_str.split('.', 1)[0]
    return code_str


def apply_soc_cleaning(df: pd.DataFrame, soc_column_candidates=None):
    if soc_column_candidates is None:
        soc_column_candidates = [c for c in df.columns if 'soc' in c.lower()]
    if not soc_column_candidates:
        raise ValueError('No SOC-like column found in dataframe')
    soc_col = soc_column_candidates[0]
    # Apply cleaning without casting the whole column to string first so real NaNs are preserved
    cleaned = df[soc_col].apply(clean_soc)
    df['soc_code'] = cleaned
    return soc_col

soc_columns = {}
for name, df in [
    ('df_occupations', df_occupations),
    ('df_skills', df_skills),
    ('df_knowledge', df_knowledge),
    ('df_education', df_education),
    ('df_activities', df_activities),
    ('df_styles', df_styles),
    ('df_abilities', df_abilities),
]:
    soc_column = apply_soc_cleaning(df)
    soc_columns[name] = soc_column
    print(f'Applied SOC cleaning to {name} using column {soc_column}')

# Example before/after for one dataframe
sample_row = df_occupations[[soc_columns.get('df_occupations', 'soc_code')]].head(3).copy()
sample_row = sample_row.rename(columns={sample_row.columns[0]: 'original_soc_column'})
sample_row['cleaned_soc_code'] = df_occupations['soc_code'].head(3).values
display(sample_row)

Applied SOC cleaning to df_occupations using column O*NET-SOC Code
Applied SOC cleaning to df_skills using column O*NET-SOC Code
Applied SOC cleaning to df_knowledge using column O*NET-SOC Code
Applied SOC cleaning to df_education using column O*NET-SOC Code
Applied SOC cleaning to df_activities using column O*NET-SOC Code
Applied SOC cleaning to df_styles using column O*NET-SOC Code
Applied SOC cleaning to df_abilities using column O*NET-SOC Code


,original_soc_column,cleaned_soc_code
0,11-1011.00,11-1011
1,11-1011.03,11-1011
2,11-1021.00,11-1021


## Section 4 — Build master_occupations.csv

Create a master occupations table by joining relevant information from the skills, education, and work styles datasets. BLS OEWS wage/employment joins are noted as a placeholder if those files are not available yet.

In [25]:
def find_column(df, keywords):
    lowered = {c.lower(): c for c in df.columns}
    for keyword in keywords:
        for lower_name, original in lowered.items():
            if keyword in lower_name:
                return original
    return None

# Identify potential join columns and feature columns
skills_soc_col = soc_columns['df_skills']
education_soc_col = soc_columns['df_education']
styles_soc_col = soc_columns['df_styles']

# Attempt to identify relevant columns in df_skills
median_wage_col = find_column(df_skills, ['median', 'wage'])
employment_col = find_column(df_skills, ['employment', 'emp'])

if median_wage_col is None or employment_col is None:
    print('Note: Median wage and employment columns were not located in df_skills. This join can be completed later with BLS OEWS data.')

# Identify education columns
typical_education_col = find_column(df_education, ['typical education', 'education'])
typical_experience_col = find_column(df_education, ['experience', 'typical experience'])

# Identify style and score columns
style_name_col = find_column(df_styles, ['work style', 'style']) or df_styles.columns[0]
style_score_col = find_column(df_styles, ['importance', 'value', 'score'])

if style_score_col is None:
    raise ValueError('Could not locate an importance score column in df_styles')

# Prepare top work style per occupation
styles_minimal = df_styles[['soc_code', style_name_col, style_score_col]].copy()
styles_minimal = styles_minimal.dropna(subset=['soc_code'])
styles_minimal['style_rank'] = styles_minimal.groupby('soc_code')[style_score_col].rank(method='first', ascending=False)
top_styles = styles_minimal[styles_minimal['style_rank'] == 1].copy()
top_styles = top_styles[['soc_code', style_name_col, style_score_col]].rename(columns={
    style_name_col: 'top_work_style',
    style_score_col: 'top_work_style_importance'
})

# Build the base occupations table
master_occupations = df_occupations.copy()

# Merge in BLS/OEWS placeholder columns if available
if median_wage_col is not None:
    master_occupations = master_occupations.merge(
        df_skills[['soc_code', median_wage_col]].rename(columns={median_wage_col: 'median_wage'}),
        on='soc_code', how='left'
    )
else:
    master_occupations['median_wage'] = np.nan

if employment_col is not None:
    master_occupations = master_occupations.merge(
        df_skills[['soc_code', employment_col]].rename(columns={employment_col: 'employment_count'}),
        on='soc_code', how='left'
    )
else:
    master_occupations['employment_count'] = np.nan

# Merge in education features
if typical_education_col is not None:
    master_occupations = master_occupations.merge(
        df_education[['soc_code', typical_education_col]].rename(columns={typical_education_col: 'typical_education_level'}),
        on='soc_code', how='left'
    )
else:
    master_occupations['typical_education_level'] = np.nan

if typical_experience_col is not None:
    master_occupations = master_occupations.merge(
        df_education[['soc_code', typical_experience_col]].rename(columns={typical_experience_col: 'typical_experience'}),
        on='soc_code', how='left'
    )
else:
    master_occupations['typical_experience'] = np.nan

# Merge top work style
master_occupations = master_occupations.merge(top_styles, on='soc_code', how='left')

print('Final master_occupations shape:', master_occupations.shape)
print('Columns:', master_occupations.columns.tolist())
print('Unique SOC codes:', master_occupations['soc_code'].nunique())
print('Null counts:')
print(master_occupations.isnull().sum().to_string())

master_occupations.to_csv(PROCESSED_DIR / 'master_occupations.csv', index=False)
print('Saved master_occupations.csv to', PROCESSED_DIR / 'master_occupations.csv')

Note: Median wage and employment columns were not located in df_skills. This join can be completed later with BLS OEWS data.
Final master_occupations shape: (1016, 10)
Columns: ['O*NET-SOC Code', 'Title', 'Description', 'soc_code', 'median_wage', 'employment_count', 'typical_education_level', 'typical_experience', 'top_work_style', 'top_work_style_importance']
Unique SOC codes: 867
Null counts:
O*NET-SOC Code                  0
Title                           0
Description                     0
soc_code                        0
median_wage                  1016
employment_count             1016
typical_education_level      1016
typical_experience           1016
top_work_style                 75
top_work_style_importance      75
Saved master_occupations.csv to ../data/processed/master_occupations.csv


## Section 5 — Build master_skills.csv

Stack the skills, knowledge, and activities datasets into a single long-format table with a common SOC code, element name, importance score, and source label.

In [27]:
def build_long_skill_df(df, source_label, name_col_keywords, score_col_keywords):
    name_col = find_column(df, name_col_keywords) or df.columns[0]
    score_col = find_column(df, score_col_keywords)
    # Fallback: if no keyword-matched score column, try the first numeric column
    if score_col is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        score_col = numeric_cols[0] if numeric_cols else None
    if score_col is None:
        raise ValueError(f'Could not locate score column for source {source_label} (tried keywords: {score_col_keywords})')
    result = df[['soc_code', name_col, score_col]].copy()
    result = result.rename(columns={name_col: 'element_name', score_col: 'importance_score'})
    result['source'] = source_label
    return result

skills_long = build_long_skill_df(df_skills, 'skill', ['skill', 'element name'], ['importance', 'relevance', 'rating'])
knowledge_long = build_long_skill_df(df_knowledge, 'knowledge', ['knowledge', 'element name'], ['importance', 'relevance', 'rating'])
activities_long = build_long_skill_df(df_activities, 'work_activity', ['activity', 'element name'], ['importance', 'relevance', 'rating'])

master_skills = pd.concat([skills_long, knowledge_long, activities_long], ignore_index=True)
master_skills = master_skills.dropna(subset=['importance_score'])

print('master_skills shape:', master_skills.shape)
print(master_skills['source'].value_counts())

master_skills.to_csv(PROCESSED_DIR / 'master_skills.csv', index=False)
print('Saved master_skills.csv to', PROCESSED_DIR / 'master_skills.csv')

master_skills shape: (150192, 4)
source
work_activity    73308
knowledge        59004
skill            17880
Name: count, dtype: int64
Saved master_skills.csv to ../data/processed/master_skills.csv


## Section 6 — Validation checks

Run a few simple checks to confirm the processed datasets are reasonable.

In [30]:
checks = []
checks.append(('master_occupations > 500 rows', master_occupations.shape[0] > 500))
checks.append(('master_skills > 10000 rows', master_skills.shape[0] > 10000))
# No duplicate SOC codes (ignore missing soc_code entries)
non_null_occ = master_occupations.dropna(subset=['soc_code'])
checks.append(('No duplicate SOC codes in master_occupations', non_null_occ['soc_code'].nunique() == non_null_occ.shape[0]))
# Ensure required sources are present (allow extra values but require these three)
required_sources = {'skill', 'knowledge', 'work_activity'}
actual_sources = set(master_skills['source'].dropna().unique())
checks.append(('All three sources appear in master_skills', required_sources.issubset(actual_sources)))
checks.append(('importance_score has no nulls in master_skills', master_skills['importance_score'].isnull().sum() == 0))

for description, passed in checks:
    print(f'{description}:', 'PASS' if passed else 'FAIL')

master_occupations > 500 rows: PASS
master_skills > 10000 rows: PASS
No duplicate SOC codes in master_occupations: FAIL
All three sources appear in master_skills: PASS
importance_score has no nulls in master_skills: PASS


## Section 7 — Quick EDA plots

Create a distribution plot for importance scores, a top-20 element frequency chart, and visualise the composition of sources in the long-format skill table.

In [31]:
plt.figure(figsize=(10, 6))
sns.histplot(master_skills['importance_score'].astype(float), bins=40, kde=False)
plt.title('Importance Score Distribution')
plt.xlabel('Importance Score')
plt.ylabel('Row count')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'importance_score_distribution.png')
plt.close()

top_20 = master_skills.groupby('element_name')['soc_code'].nunique().nlargest(20).sort_values(ascending=False)
plt.figure(figsize=(12, 8))
sns.barplot(x=top_20.values, y=top_20.index, palette='viridis')
plt.title('Top 20 Most Common Elements Across Occupations')
plt.xlabel('Number of unique SOC codes')
plt.ylabel('Element name')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'top_20_elements.png')
plt.close()

source_counts = master_skills['source'].value_counts()
plt.figure(figsize=(8, 6))
sns.barplot(x=source_counts.index, y=source_counts.values, palette='pastel')
plt.title('Counts by Source in master_skills')
plt.xlabel('Source')
plt.ylabel('Row count')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'source_counts.png')
plt.close()

print('Saved plot files to', PROCESSED_DIR)

Saved plot files to ../data/processed


/var/folders/y4/lh8fczc90wl0bvv1dm11w49m0000gn/T/ipykernel_79741/2022177680.py:12: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_20.values, y=top_20.index, palette='viridis')
/var/folders/y4/lh8fczc90wl0bvv1dm11w49m0000gn/T/ipykernel_79741/2022177680.py:22: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=source_counts.index, y=source_counts.values, palette='pastel')
